In [1]:

# ============================================================
# Step 1: Load Mushroom Dataset — Orange Comparison
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# Project directories
PROJECT_DIR = Path.cwd()
SPLIT_DIR = PROJECT_DIR / "outputs" / "data_split"
TRAINING_DIR = PROJECT_DIR / "outputs" / "training"

# Separate folder for Orange results
ORANGE_DIR = PROJECT_DIR / "reproducibility_orange"
ORANGE_DIR.mkdir(parents=True, exist_ok=True)

# Load AVISTA's saved data partitions
data = {
    "X_train": np.load(SPLIT_DIR / "X_train.npy"),
    "y_train": np.load(SPLIT_DIR / "y_train_encoded.npy"),

    "X_train_balanced": np.load(
        SPLIT_DIR / "X_train_balanced.npy"
    ),
    "y_train_balanced": np.load(
        SPLIT_DIR / "y_train_balanced_encoded.npy"
    ),

    "X_val": np.load(SPLIT_DIR / "X_val.npy"),
    "y_val": np.load(SPLIT_DIR / "y_val_encoded.npy"),

    "X_test": np.load(SPLIT_DIR / "X_test.npy"),
    "y_test": np.load(SPLIT_DIR / "y_test_encoded.npy"),
}

# Verify all partitions
for split in ["train", "val", "test"]:
    assert len(data[f"X_{split}"]) == len(data[f"y_{split}"])

assert len(data["X_train_balanced"]) == len(data["y_train_balanced"])

# Display shapes and class distributions
print("PROJECT:", PROJECT_DIR.name)
print("Orange results:", ORANGE_DIR)
print("=" * 65)

for split in ["train", "val", "test"]:
    X = data[f"X_{split}"]
    y = data[f"y_{split}"]

    classes, counts = np.unique(y, return_counts=True)

    print(f"\n{split.upper()}")
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("Class distribution:", dict(zip(classes.tolist(), counts.tolist())))

print("\nBALANCED TRAINING")
print("X shape:", data["X_train_balanced"].shape)
print("y shape:", data["y_train_balanced"].shape)

print("\nData loaded successfully.")

PROJECT: mushroom
Orange results: d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\mushroom\reproducibility_orange

TRAIN
X shape: (5686, 67)
y shape: (5686,)
Class distribution: {0: 2943, 1: 2743}

VAL
X shape: (813, 67)
y shape: (813,)
Class distribution: {0: 422, 1: 391}

TEST
X shape: (1625, 67)
y shape: (1625,)
Class distribution: {0: 843, 1: 782}

BALANCED TRAINING
X shape: (5686, 67)
y shape: (5686,)

Data loaded successfully.


In [3]:
%pip install orange3

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached keyring-25.7.0-py3-none-any.whl.metadata (21 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached QtPy-2.4.3-py3-none-any.whl.metadata (12 kB)
  Using cached pywin32_ctypes-0.2.3-py3-none-any.whl.metadata (3.9 kB)
  Using cached jaraco.classes-3.4.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached jaraco_functools-4.6.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached jaraco_context-6.1.2-py3-none-any.whl.metadata (4.2 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------- ----------------- 2.1/3.7 MB 16.8 MB/s eta 0:00:01
   ------------

  DEPRECATION: Building 'baycomp' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'baycomp'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'python-louvain' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'python-louvain'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'serverfiles' using the legacy setup.py bdist_

In [5]:
import sys
import Orange
from Orange.data import Table, Domain, ContinuousVariable, DiscreteVariable

print("Python environment:", sys.executable)
print("Orange version:", Orange.__version__)
print("Orange imported successfully.")

Python environment: c:\Users\monzu\anaconda3\envs\mimi\python.exe
Orange version: 3.40.0
Orange imported successfully.


In [6]:
# ============================================================
# Step 2: Convert AVISTA Arrays to Orange Tables
# ============================================================

import Orange
from Orange.data import (
    Table,
    Domain,
    ContinuousVariable,
    DiscreteVariable
)

# 1. Define the Orange domain
n_features = data["X_train"].shape[1]

features = [
    ContinuousVariable(f"feature_{i+1}")
    for i in range(n_features)
]

target = DiscreteVariable(
    "mushroom_class",
    values=["e", "p"]  # 0 = edible, 1 = poisonous
)

orange_domain = Domain(features, target)

# 2. Conversion helper


def to_orange_table(X, y):
    return Table.from_numpy(
        orange_domain,
        X=np.asarray(X, dtype=np.float64),
        Y=np.asarray(y, dtype=np.float64)
    )


# 3. Convert all four partitions
orange_train = to_orange_table(
    data["X_train"], data["y_train"]
)

orange_train_balanced = to_orange_table(
    data["X_train_balanced"], data["y_train_balanced"]
)

orange_val = to_orange_table(
    data["X_val"], data["y_val"]
)

orange_test = to_orange_table(
    data["X_test"], data["y_test"]
)

# 4. Verify conversion
print("Orange version:", Orange.__version__)
print("=" * 65)

for name, table in {
    "Train": orange_train,
    "Balanced Train": orange_train_balanced,
    "Validation": orange_val,
    "Test": orange_test
}.items():
    print(f"{name:<18} Rows={len(table):<6} Features={len(table.domain.attributes)}")

print("\nTarget:", orange_domain.class_var)
print("Target categories:", orange_domain.class_var.values)

assert np.array_equal(orange_train.X, data["X_train"])
assert np.array_equal(orange_test.X, data["X_test"])
assert np.array_equal(orange_test.Y, data["y_test"])

print("\nOrange tables created and verified successfully.")

Orange version: 3.40.0
Train              Rows=5686   Features=67
Balanced Train     Rows=5686   Features=67
Validation         Rows=813    Features=67
Test               Rows=1625   Features=67

Target: mushroom_class
Target categories: ('e', 'p')

Orange tables created and verified successfully.


In [7]:

# ============================================================
# Step 3: Orange Logistic Regression — Mushroom
# AVISTA vs. Orange Python API
# ============================================================

import json
import numpy as np
import pandas as pd

from Orange.classification import LogisticRegressionLearner
from Orange.base import Model

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Load AVISTA reference results
# ------------------------------------------------------------

with open(
    TRAINING_DIR / "training_results.json",
    "r",
    encoding="utf-8"
) as f:
    avista_results = json.load(f)

lr_reference = next(
    result
    for result in avista_results["results"]
    if "logistic regression" in result["model_name"].lower()
)

# ------------------------------------------------------------
# 2. Calculate effective class weights
# ------------------------------------------------------------

y_train = data["y_train_balanced"]

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    int(c): float(w)
    for c, w in zip(classes, weights)
}

print("ORANGE LOGISTIC REGRESSION")
print("=" * 70)
print("Orange version:", Orange.__version__)
print("Class weights:", class_weights)
print("Training samples:", len(orange_train_balanced))

# ------------------------------------------------------------
# 3. Configure and train Orange learner
# ------------------------------------------------------------

# Empty preprocessors: AVISTA already supplied
# preprocessed features, so avoid extra transformations.

orange_lr_learner = LogisticRegressionLearner(
    penalty="l2",
    C=1.0,
    tol=0.0001,
    solver="lbfgs",
    max_iter=100,
    fit_intercept=True,
    class_weight=class_weights,
    random_state=None,
    preprocessors=[]
)

orange_lr_model = orange_lr_learner(
    orange_train_balanced
)

print("\nTraining completed.")
print("Underlying estimator:", type(orange_lr_model.skl_model))
print("Iterations:", orange_lr_model.skl_model.n_iter_)

# ------------------------------------------------------------
# 4. Evaluate Orange model
# ------------------------------------------------------------


def evaluate_orange_lr(model, table):

    predictions = np.asarray(
        model(table, Model.Value)
    ).astype(int)

    probabilities = np.asarray(
        model(table, Model.Probs)
    )

    actual = np.asarray(table.Y).astype(int)

    metrics = {
        "accuracy": accuracy_score(actual, predictions),
        "macro_f1": f1_score(
            actual, predictions, average="macro"
        ),
        "weighted_f1": f1_score(
            actual, predictions, average="weighted"
        ),
        "balanced_accuracy": balanced_accuracy_score(
            actual, predictions
        ),
        "macro_precision": precision_score(
            actual, predictions, average="macro",
            zero_division=0
        ),
        "macro_recall": recall_score(
            actual, predictions, average="macro",
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            actual, probabilities[:, 1]
        )
    }

    return metrics, predictions

# ------------------------------------------------------------
# 5. Compare with AVISTA
# ------------------------------------------------------------


tables = {
    "Train": orange_train_balanced,
    "Validation": orange_val,
    "Test": orange_test
}

comparison_rows = {}

for split, table in tables.items():

    orange_metrics, predictions = evaluate_orange_lr(
        orange_lr_model, table
    )

    reference_metrics = lr_reference[
        f"{split.lower()}_metrics"
    ]

    comparison_rows[split] = [
        {
            "Dataset": PROJECT_DIR.name,
            "Model": "Logistic Regression",
            "Split": split,
            "Metric": metric,
            "AVISTA": reference_metrics.get(metric, np.nan),
            "Orange": value,
            "Absolute Difference": abs(
                value - reference_metrics.get(metric, np.nan)
            )
        }
        for metric, value in orange_metrics.items()
    ]

orange_lr_df = pd.DataFrame(
    [
        row
        for rows in comparison_rows.values()
        for row in rows
    ]
)

for split in tables:
    print(f"\n{split.upper()} COMPARISON")
    display(
        orange_lr_df[
            orange_lr_df["Split"] == split
        ][[
            "Metric",
            "AVISTA",
            "Orange",
            "Absolute Difference"
        ]].round(8)
    )

# ------------------------------------------------------------
# 6. Test confusion matrix
# ------------------------------------------------------------

test_metrics, test_predictions = evaluate_orange_lr(
    orange_lr_model,
    orange_test
)

print("\nAVISTA test confusion matrix:")
print(
    np.asarray(
        lr_reference["test_metrics"]["confusion_matrix"]
    )
)

print("\nOrange test confusion matrix:")
print(
    confusion_matrix(
        data["y_test"],
        test_predictions,
        labels=[0, 1]
    )
)

# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------

ORANGE_DIR.mkdir(parents=True, exist_ok=True)

output_path = (
    ORANGE_DIR / "LR_orange_reproducibility_results.csv"
)

orange_lr_df.to_csv(output_path, index=False)

print("\nResults saved to:")
print(output_path)

ORANGE LOGISTIC REGRESSION
Orange version: 3.40.0
Class weights: {0: 0.9660210669384981, 1: 1.0364564345607}
Training samples: 5686

Training completed.
Underlying estimator: <class 'sklearn.linear_model._logistic.LogisticRegression'>
Iterations: [35]

TRAIN COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
0,accuracy,0.985051,0.985051,0.0
1,macro_f1,0.985040,0.985040,0.0
2,weighted_f1,0.985054,0.985054,0.0
3,balanced_accuracy,0.985324,0.985324,0.0
4,macro_precision,0.984892,0.984892,0.0
5,macro_recall,0.985324,0.985324,0.0
6,roc_auc,0.999489,0.999489,0.0



VALIDATION COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
7,accuracy,0.986470,0.986470,0.0
8,macro_f1,0.986449,0.986449,0.0
9,weighted_f1,0.986469,0.986469,0.0
10,balanced_accuracy,0.986403,0.986403,0.0
11,macro_precision,0.986498,0.986498,0.0
12,macro_recall,0.986403,0.986403,0.0
13,roc_auc,0.999321,0.999321,0.0



TEST COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
14,accuracy,0.976000,0.976000,0.0
15,macro_f1,0.975988,0.975988,0.0
16,weighted_f1,0.976008,0.976008,0.0
17,balanced_accuracy,0.976544,0.976544,0.0
18,macro_precision,0.975895,0.975895,0.0
19,macro_recall,0.976544,0.976544,0.0
20,roc_auc,0.999273,0.999273,0.0



AVISTA test confusion matrix:
[[811  32]
 [  7 775]]

Orange test confusion matrix:
[[811  32]
 [  7 775]]

Results saved to:
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\mushroom\reproducibility_orange\LR_orange_reproducibility_results.csv


In [8]:

# ============================================================
# Step 4: Orange Random Forest — Mushroom
# AVISTA vs. Orange Python API
# ============================================================

import json
import inspect
import numpy as np
import pandas as pd

from Orange.classification import RandomForestLearner
from Orange.base import Model
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, balanced_accuracy_score,
    roc_auc_score, confusion_matrix
)

# ------------------------------------------------------------
# 1. Load AVISTA reference and saved configuration
# ------------------------------------------------------------

rf_reference = next(
    result for result in avista_results["results"]
    if "random forest" in result["model_name"].lower()
)

rf_config_path = (
    TRAINING_DIR / "RandomForest" / "model_config.json"
)

with open(rf_config_path, "r", encoding="utf-8") as f:
    rf_config = json.load(f)

# ------------------------------------------------------------
# 2. Reconstruct effective model parameters
# ------------------------------------------------------------

y_fit = np.asarray(data["y_train_balanced"])
classes = np.unique(y_fit)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_fit
)

class_weights = {
    int(c): float(w)
    for c, w in zip(classes, weights)
}

rf_params = rf_config["parameters"].copy()

# Same effective weighting strategy identified in AVISTA
rf_params["class_weight"] = class_weights
rf_params["random_state"] = rf_config.get(
    "random_state",
    rf_params.get("random_state", 42)
)

# Orange exposes a subset of sklearn RF constructor options.
supported = set(
    inspect.signature(RandomForestLearner).parameters
) - {"self", "preprocessors"}

unsupported = {
    key: value for key, value in rf_params.items()
    if key not in supported
}

orange_rf_params = {
    key: value for key, value in rf_params.items()
    if key in supported
}

print("ORANGE RANDOM FOREST")
print("=" * 70)
print("Orange version:", Orange.__version__)
print("Effective class weights:", class_weights)
print("Orange RF parameters:", orange_rf_params)
print("Parameters not exposed by Orange:", unsupported)

# ------------------------------------------------------------
# 3. Train through Orange
# ------------------------------------------------------------

orange_rf_learner = RandomForestLearner(
    **orange_rf_params,
    preprocessors=[]
)

orange_rf_model = orange_rf_learner(
    orange_train_balanced
)

print("\nTraining completed.")
print("Underlying estimator:", type(orange_rf_model.skl_model))
print("Trees trained:", len(orange_rf_model.skl_model.estimators_))

# ------------------------------------------------------------
# 4. Evaluation function
# ------------------------------------------------------------


def evaluate_orange_rf(model, table):

    actual = np.asarray(table.Y).astype(int)

    predictions = np.asarray(
        model(table, Model.Value)
    ).astype(int)

    probabilities = np.asarray(
        model(table, Model.Probs)
    )

    metrics = {
        "accuracy": accuracy_score(actual, predictions),
        "macro_f1": f1_score(
            actual, predictions, average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            actual, predictions, average="weighted",
            zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            actual, predictions
        ),
        "macro_precision": precision_score(
            actual, predictions, average="macro",
            zero_division=0
        ),
        "macro_recall": recall_score(
            actual, predictions, average="macro",
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            actual, probabilities[:, 1]
        )
    }

    return metrics, predictions

# ------------------------------------------------------------
# 5. Train / Validation / Test comparisons
# ------------------------------------------------------------


tables = {
    "Train": orange_train_balanced,
    "Validation": orange_val,
    "Test": orange_test
}

rows = []
rf_predictions = {}

for split, table in tables.items():

    orange_metrics, predictions = evaluate_orange_rf(
        orange_rf_model, table
    )

    rf_predictions[split] = predictions

    reference_metrics = rf_reference[
        f"{split.lower()}_metrics"
    ]

    for metric, value in orange_metrics.items():

        avista_value = reference_metrics.get(
            metric, np.nan
        )

        rows.append({
            "Dataset": PROJECT_DIR.name,
            "Model": "Random Forest",
            "Split": split,
            "Metric": metric,
            "AVISTA": avista_value,
            "Orange": value,
            "Absolute Difference": abs(
                value - avista_value
            )
        })

orange_rf_df = pd.DataFrame(rows)

for split in tables:

    print(f"\n{split.upper()} COMPARISON")
    display(
        orange_rf_df[
            orange_rf_df["Split"] == split
        ][[
            "Metric", "AVISTA",
            "Orange", "Absolute Difference"
        ]].round(8)
    )

# ------------------------------------------------------------
# 6. Compare test confusion matrices
# ------------------------------------------------------------

avista_cm = np.asarray(
    rf_reference["test_metrics"]["confusion_matrix"]
)

orange_cm = confusion_matrix(
    data["y_test"],
    rf_predictions["Test"],
    labels=[0, 1]
)

print("\nAVISTA test confusion matrix:")
print(avista_cm)

print("\nOrange test confusion matrix:")
print(orange_cm)

print("\nConfusion matrices identical:",
      np.array_equal(avista_cm, orange_cm))

# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------

ORANGE_DIR.mkdir(parents=True, exist_ok=True)

rf_output_path = (
    ORANGE_DIR / "RF_orange_reproducibility_results.csv"
)

orange_rf_df.to_csv(
    rf_output_path,
    index=False
)

print("\nResults saved to:")
print(rf_output_path)

ORANGE RANDOM FOREST
Orange version: 3.40.0
Effective class weights: {0: 0.9660210669384981, 1: 1.0364564345607}
Orange RF parameters: {'n_estimators': 100, 'criterion': 'gini', 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'min_weight_fraction_leaf': 0.0, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'bootstrap': True, 'oob_score': False, 'n_jobs': None, 'verbose': 0, 'class_weight': {0: 0.9660210669384981, 1: 1.0364564345607}, 'random_state': 42}
Parameters not exposed by Orange: {'min_impurity_decrease': 0.0, 'warm_start': False, 'ccp_alpha': 0.0, 'max_samples': None, 'monotonic_cst': None}

Training completed.
Underlying estimator: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Trees trained: 100

TRAIN COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
0,accuracy,1.0,1.0,0.0
1,macro_f1,1.0,1.0,0.0
2,weighted_f1,1.0,1.0,0.0
3,balanced_accuracy,1.0,1.0,0.0
4,macro_precision,1.0,1.0,0.0
5,macro_recall,1.0,1.0,0.0
6,roc_auc,1.0,1.0,0.0



VALIDATION COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
7,accuracy,1.0,1.0,0.0
8,macro_f1,1.0,1.0,0.0
9,weighted_f1,1.0,1.0,0.0
10,balanced_accuracy,1.0,1.0,0.0
11,macro_precision,1.0,1.0,0.0
12,macro_recall,1.0,1.0,0.0
13,roc_auc,1.0,1.0,0.0



TEST COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
14,accuracy,1.0,1.0,0.0
15,macro_f1,1.0,1.0,0.0
16,weighted_f1,1.0,1.0,0.0
17,balanced_accuracy,1.0,1.0,0.0
18,macro_precision,1.0,1.0,0.0
19,macro_recall,1.0,1.0,0.0
20,roc_auc,1.0,1.0,0.0



AVISTA test confusion matrix:
[[843   0]
 [  0 782]]

Orange test confusion matrix:
[[843   0]
 [  0 782]]

Confusion matrices identical: True

Results saved to:
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\mushroom\reproducibility_orange\RF_orange_reproducibility_results.csv


In [9]:

# ============================================================
# Step 5: Orange XGBoost — Mushroom
# AVISTA vs. Orange Python API
# ============================================================

import json
import inspect
import numpy as np
import pandas as pd
import Orange
import xgboost as xgb

from Orange.classification.xgb import XGBClassifier as OrangeXGBClassifier
from Orange.base import Model

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# 1. LOAD AVISTA REFERENCE AND CONFIGURATION
# ============================================================

xgb_reference = next(
    result
    for result in avista_results["results"]
    if "xgboost" in result["model_name"].lower()
)

xgb_config_path = (
    TRAINING_DIR / "XGBoost" / "model_config.json"
)

with open(xgb_config_path, "r", encoding="utf-8") as f:
    xgb_config = json.load(f)

xgb_params = xgb_config["parameters"].copy()

xgb_params["random_state"] = xgb_config.get(
    "random_state",
    xgb_params.get("random_state", 42)
)

print("=" * 75)
print("ORANGE XGBOOST — REPRODUCIBILITY VALIDATION")
print("=" * 75)

print("Dataset:", PROJECT_DIR.name)
print("Orange version:", Orange.__version__)
print("XGBoost version:", xgb.__version__)
print("Training samples:", len(orange_train_balanced))

# ============================================================
# 2. RECONSTRUCT SUPPORTED ORANGE PARAMETERS
# ============================================================

# Orange's XGBoost wrapper exposes most, but not
# necessarily all, parameters of xgboost.XGBClassifier.

signature = inspect.signature(OrangeXGBClassifier)

supported = set(signature.parameters) - {
    "self", "preprocessors"
}

orange_xgb_params = {
    key: value
    for key, value in xgb_params.items()
    if key in supported
}

unsupported_params = {
    key: value
    for key, value in xgb_params.items()
    if key not in supported
}

print("\nAVISTA model parameters:")
print(xgb_params)

print("\nOrange-supported parameters:")
print(orange_xgb_params)

print("\nParameters not directly exposed by Orange:")
print(
    unsupported_params
    if unsupported_params
    else "None"
)

# Note:
# AVISTA's XGBoost training calls fit(X, y)
# without class_weight or sample_weight.
# We do not add class weighting here.

# ============================================================
# 3. TRAIN ORANGE XGBOOST
# ============================================================

orange_xgb_learner = OrangeXGBClassifier(
    **orange_xgb_params,
    preprocessors=[]
)

orange_xgb_model = orange_xgb_learner(
    orange_train_balanced
)

print("\nTraining completed.")
print(
    "Underlying estimator:",
    type(orange_xgb_model.skl_model)
)

print(
    "Number of estimators:",
    orange_xgb_model.skl_model.n_estimators
)

# ============================================================
# 4. EVALUATION FUNCTION
# ============================================================


def evaluate_orange_xgb(model, table):

    actual = np.asarray(table.Y).astype(int)

    predictions = np.asarray(
        model(table, Model.Value)
    ).astype(int)

    probabilities = np.asarray(
        model(table, Model.Probs)
    )

    metrics = {
        "accuracy": accuracy_score(
            actual, predictions
        ),

        "macro_f1": f1_score(
            actual,
            predictions,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            actual,
            predictions,
            average="weighted",
            zero_division=0
        ),

        "balanced_accuracy": balanced_accuracy_score(
            actual, predictions
        ),

        "macro_precision": precision_score(
            actual,
            predictions,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            actual,
            predictions,
            average="macro",
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            actual,
            probabilities[:, 1]
        )
    }

    return metrics, predictions, probabilities


# ============================================================
# 5. TRAIN / VALIDATION / TEST COMPARISON
# ============================================================

tables = {
    "Train": orange_train_balanced,
    "Validation": orange_val,
    "Test": orange_test
}

rows = []
orange_xgb_predictions = {}

for split, table in tables.items():

    orange_metrics, predictions, probabilities = (
        evaluate_orange_xgb(
            orange_xgb_model,
            table
        )
    )

    orange_xgb_predictions[split] = predictions

    reference_metrics = xgb_reference[
        f"{split.lower()}_metrics"
    ]

    for metric, orange_value in orange_metrics.items():

        avista_value = reference_metrics.get(
            metric, np.nan
        )

        rows.append({
            "Dataset": PROJECT_DIR.name,
            "Model": "XGBoost",
            "Split": split,
            "Metric": metric,
            "AVISTA": avista_value,
            "Orange": orange_value,
            "Signed Difference": (
                orange_value - avista_value
            ),
            "Absolute Difference": abs(
                orange_value - avista_value
            )
        })

orange_xgb_df = pd.DataFrame(rows)

for split in tables:

    print("\n" + "=" * 75)
    print(f"{split.upper()} COMPARISON")
    print("=" * 75)

    display(
        orange_xgb_df[
            orange_xgb_df["Split"] == split
        ][[
            "Metric",
            "AVISTA",
            "Orange",
            "Absolute Difference"
        ]].round(8)
    )


# ============================================================
# 6. TEST CONFUSION MATRIX
# ============================================================

avista_cm = np.asarray(
    xgb_reference["test_metrics"]["confusion_matrix"]
)

orange_cm = confusion_matrix(
    data["y_test"],
    orange_xgb_predictions["Test"],
    labels=[0, 1]
)

print("\n" + "=" * 75)
print("TEST CONFUSION MATRICES")
print("=" * 75)

print("\nAVISTA:")
print(avista_cm)

print("\nOrange:")
print(orange_cm)

print(
    "\nConfusion matrices identical:",
    np.array_equal(avista_cm, orange_cm)
)

print("\nOrange classification report:")
print(
    classification_report(
        data["y_test"],
        orange_xgb_predictions["Test"],
        target_names=["Edible", "Poisonous"],
        zero_division=0
    )
)


# ============================================================
# 7. SAVE RESULTS
# ============================================================

ORANGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

xgb_output_path = (
    ORANGE_DIR / "XGB_orange_reproducibility_results.csv"
)

orange_xgb_df.to_csv(
    xgb_output_path,
    index=False
)

# Save configuration differences separately
config_output_path = (
    ORANGE_DIR / "XGB_orange_configuration_audit.json"
)

with open(config_output_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "dataset": PROJECT_DIR.name,
            "orange_version": Orange.__version__,
            "xgboost_version": xgb.__version__,
            "avista_parameters": xgb_params,
            "orange_parameters": orange_xgb_params,
            "unsupported_parameters": unsupported_params
        },
        f,
        indent=2,
        default=str
    )

print("\n" + "=" * 75)
print("RESULTS SAVED")
print("=" * 75)

print(xgb_output_path)
print(config_output_path)

ORANGE XGBOOST — REPRODUCIBILITY VALIDATION
Dataset: mushroom
Orange version: 3.40.0
XGBoost version: 3.0.2
Training samples: 5686

AVISTA model parameters:
{'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'binary:logistic', 'eval_metric': 'mlogloss', 'tree_method': 'auto', 'booster': 'gbtree', 'gamma': 0.0, 'min_child_weight': 1.0, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'scale_pos_weight': 1.0, 'random_state': 42, 'n_jobs': None, 'verbosity': 0, 'enable_categorical': False}

Orange-supported parameters:
{'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'binary:logistic', 'tree_method': 'auto', 'booster': 'gbtree', 'gamma': 0.0, 'min_child_weight': 1.0, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'scale_pos_weight': 1.0, 'random_state': 42, 'n_jobs': None, 'verbosity': 0}

Parameters not directly exposed by Orange:
{'eval_metric': 'mlogloss', 'enable_categorical

,Metric,AVISTA,Orange,Absolute Difference
0,accuracy,1.0,1.0,0.0
1,macro_f1,1.0,1.0,0.0
2,weighted_f1,1.0,1.0,0.0
3,balanced_accuracy,1.0,1.0,0.0
4,macro_precision,1.0,1.0,0.0
5,macro_recall,1.0,1.0,0.0
6,roc_auc,1.0,1.0,0.0



VALIDATION COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
7,accuracy,1.0,1.0,0.0
8,macro_f1,1.0,1.0,0.0
9,weighted_f1,1.0,1.0,0.0
10,balanced_accuracy,1.0,1.0,0.0
11,macro_precision,1.0,1.0,0.0
12,macro_recall,1.0,1.0,0.0
13,roc_auc,1.0,1.0,0.0



TEST COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
14,accuracy,1.0,1.0,0.0
15,macro_f1,1.0,1.0,0.0
16,weighted_f1,1.0,1.0,0.0
17,balanced_accuracy,1.0,1.0,0.0
18,macro_precision,1.0,1.0,0.0
19,macro_recall,1.0,1.0,0.0
20,roc_auc,1.0,1.0,0.0



TEST CONFUSION MATRICES

AVISTA:
[[843   0]
 [  0 782]]

Orange:
[[843   0]
 [  0 782]]

Confusion matrices identical: True

Orange classification report:
              precision    recall  f1-score   support

      Edible       1.00      1.00      1.00       843
   Poisonous       1.00      1.00      1.00       782

    accuracy                           1.00      1625
   macro avg       1.00      1.00      1.00      1625
weighted avg       1.00      1.00      1.00      1625


RESULTS SAVED
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\mushroom\reproducibility_orange\XGB_orange_reproducibility_results.csv
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\mushroom\reproducibility_orange\XGB_orange_configuration_audit.json


In [10]:

# ============================================================
# Step 6: Orange RF and XGBoost — Final verification
# ============================================================

for name, df in [
    ("Random Forest", orange_rf_df),
    ("XGBoost", orange_xgb_df)
]:
    print(f"\n{'=' * 70}")
    print(f"{name.upper()} — TEST RESULTS")
    print("=" * 70)

    test = df[
        (df["Split"] == "Test") &
        (df["Metric"].isin(["accuracy", "macro_f1", "roc_auc"]))
    ]

    display(test[
        ["Metric", "AVISTA", "Orange", "Absolute Difference"]
    ].round(10))

    print(
        "Maximum absolute difference across all reported metrics:",
        df["Absolute Difference"].max()
    )

print("\nXGBOOST EFFECTIVE PARAMETERS")
print("=" * 70)

xgb_effective = orange_xgb_model.skl_model.get_params()

for key in ["eval_metric", "enable_categorical",
            "n_estimators", "random_state"]:
    print(f"{key}: {xgb_effective.get(key)}")

print("\nRANDOM FOREST EFFECTIVE PARAMETERS")
print("=" * 70)

rf_effective = orange_rf_model.skl_model.get_params()

for key in [
    "min_impurity_decrease", "warm_start",
    "ccp_alpha", "max_samples", "monotonic_cst",
    "class_weight", "random_state"
]:
    print(f"{key}: {rf_effective.get(key)}")

print("\nSAVED FILES")
print("=" * 70)

for filename in [
    "LR_orange_reproducibility_results.csv",
    "RF_orange_reproducibility_results.csv",
    "XGB_orange_reproducibility_results.csv"
]:
    path = ORANGE_DIR / filename
    print(f"{filename}: {'Found' if path.exists() else 'Missing'}")


RANDOM FOREST — TEST RESULTS


,Metric,AVISTA,Orange,Absolute Difference
14,accuracy,1.0,1.0,0.0
15,macro_f1,1.0,1.0,0.0
20,roc_auc,1.0,1.0,0.0


Maximum absolute difference across all reported metrics: 0.0

XGBOOST — TEST RESULTS


,Metric,AVISTA,Orange,Absolute Difference
14,accuracy,1.0,1.0,0.0
15,macro_f1,1.0,1.0,0.0
20,roc_auc,1.0,1.0,0.0


Maximum absolute difference across all reported metrics: 0.0

XGBOOST EFFECTIVE PARAMETERS
eval_metric: None
enable_categorical: False
n_estimators: 300
random_state: 42

RANDOM FOREST EFFECTIVE PARAMETERS
min_impurity_decrease: 0.0
warm_start: False
ccp_alpha: 0.0
max_samples: None
monotonic_cst: None
class_weight: {0: 0.9660210669384981, 1: 1.0364564345607}
random_state: 42

SAVED FILES
LR_orange_reproducibility_results.csv: Found
RF_orange_reproducibility_results.csv: Found
XGB_orange_reproducibility_results.csv: Found


In [11]:
# ============================================================
# Step 7: Consolidate Orange Comparison Results — Mushroom
# ============================================================

import pandas as pd

all_orange_results = pd.concat(
    [
        orange_lr_df,
        orange_rf_df,
        orange_xgb_df
    ],
    ignore_index=True
)

# Consolidated full results
all_orange_results.to_csv(
    ORANGE_DIR / "Orange_all_models_comparison.csv",
    index=False
)

# Publication-ready test summary
test_summary = (
    all_orange_results[
        (all_orange_results["Split"] == "Test") &
        (all_orange_results["Metric"].isin(
            ["accuracy", "macro_f1", "roc_auc"]
        ))
    ]
    .pivot_table(
        index=["Dataset", "Model"],
        columns="Metric",
        values=["AVISTA", "Orange"],
        aggfunc="first"
    )
)

test_summary.columns = [
    f"{platform}_{metric}"
    for platform, metric in test_summary.columns
]

test_summary = test_summary.reset_index()

display(test_summary.round(6))

test_summary.to_csv(
    ORANGE_DIR / "Orange_test_summary.csv",
    index=False
)

print("Consolidated Orange results saved.")

,Dataset,Model,AVISTA_accuracy,AVISTA_macro_f1,AVISTA_roc_auc,Orange_accuracy,Orange_macro_f1,Orange_roc_auc
0,mushroom,Logistic Regression,0.976,0.975988,0.999273,0.976,0.975988,0.999273
1,mushroom,Random Forest,1.000,1.000000,1.000000,1.000,1.000000,1.000000
2,mushroom,XGBoost,1.000,1.000000,1.000000,1.000,1.000000,1.000000


Consolidated Orange results saved.
